## Exploratory Data Analysis
- In this notebook, I want to understand what my data actually looks like
- I want to determine which columns to keep or drop for my analysis
- I want to verify data types, null counts, and any other inconsistencies that need to be addressed in the cleaning notebook

## Step 1:
- I want to first combine each dataset's years into their respective 
  commodities, to make analyzing the data easier and faster
- This makes sense to do because all 3 years of a given commodity share the same structure — they only differ by year

In [5]:
import pandas as pd

commodity_files = {
    "Avocados": ["avocados_23.csv", "avocados_24.csv", "avocados_25.csv"],
    "Tomatoes": ["tomatoes_23.csv", "tomatoes_24.csv", "tomatoes_25.csv"],
    "Romaine Lettuce": ["lettuceRomaine_23.csv", "lettuceRomaine_24.csv", "lettuceRomaine_25.csv"],
    "Strawberries": ["strawberries_23.csv", "strawberries_24.csv", "strawberries_25.csv"],
    "Bananas": ["bananas_23.csv", "bananas_24.csv", "bananas_25.csv"],
}

## Step 2:
- I combined each commodity's 3 yearly files into one dataframe
- I stored each combined dataframe in a dictionary (raw_combined), keyed by 
  commodity name, so I can access and explore each one separately
- I also converted report_date to a proper date type here, since I'll need 
  it for date-based checks later

In [8]:
raw_combined = {}

for name, files in commodity_files.items():
    dfs = [pd.read_csv(f'../data/raw/{f}') for f in files]
    combined = pd.concat(dfs, ignore_index=True)
    combined['report_date'] = pd.to_datetime(combined['report_date'])
    raw_combined[name] = combined
    print(f"{name}: {combined.shape[0]} rows, {combined['report_date'].nunique()} unique dates")

Avocados: 14940 rows, 745 unique dates
Tomatoes: 11442 rows, 744 unique dates
Romaine Lettuce: 4251 rows, 744 unique dates
Strawberries: 7837 rows, 745 unique dates
Bananas: 22366 rows, 745 unique dates


## Step 3:
- Now that each commodity's years are combined, I want to run .info() on 
  all 5 at once
- This lets me check the columns, data types, and null counts for every 
  commodity in one pass
- I'm doing this to confirm the columns and issues I find are consistent 
  across all 5 commodities, not just specific to one

In [ ]:
for name, df in raw_combined.items():
    print(f"\n===== {name} =====")
    df.info()

Avocados: 14940 rows, 745 unique dates
Tomatoes: 11442 rows, 744 unique dates
Romaine Lettuce: 4251 rows, 744 unique dates
Strawberries: 7837 rows, 745 unique dates
Bananas: 22366 rows, 745 unique dates

===== Avocados =====
<class 'pandas.DataFrame'>
RangeIndex: 14940 entries, 0 to 14939
Data columns (total 29 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   report_date           14940 non-null  datetime64[us]
 1   location              14940 non-null  str           
 2   group                 14940 non-null  str           
 3   commodity             14940 non-null  str           
 4   variety               14940 non-null  str           
 5   package               14940 non-null  str           
 6   grade                 0 non-null      float64       
 7   origin                14940 non-null  str           
 8   district              0 non-null      float64       
 9   item_size             14940 non-nu

## Findings:
- All 5 commodities have the same 29 columns, since they all come from 
  the same USDA report format
- 12 columns are completely empty for every commodity (grade, district, 
  environment, unit_sales, storage, crop, repack, transportation_mode, 
  properties, market_tone_comments, offerings_comments, commodity_comments) 
  ... these can be dropped
- quality, condition, and appearance are populated in under 15% of rows 
  across all commodities ... too sparse to use, these can be dropped too
- low_price and high_price are populated in almost every row (~97%+)
- mostly_low_price and mostly_high_price are only populated in about 
  55% of rows ... meaning ~45% are missing
- Since mostly_price is more accurate but isn't always available, I'll use 
  it when present and fall back to the low/high midpoint when it's missing
- organic is fully populated (Y/N) ... I'll filter to conventional (N) only, 
  since organic listings tend to carry a price premium that would skew 
  the results
- report_date is currently stored as text, not a date ... this whas fixed above
- Additonal findings may arise during the cleaning of these data sets